# 02 · QC semantic và ba checkpoint semantic

Xuất `Segmentation mask 1.1` từ CVAT; để ZIP vào `submissions/<mã_task>.zip`. Ô kiểm chỉ kiểm cấu trúc, ảnh và labelmap, **không so đáp án**. Không cần chạy ô này để vẽ được trên CVAT.

In [ ]:
from pathlib import Path
import sys
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/manifest.json").is_file()), None)
assert root is not None, "Mở notebook từ thư mục repo Day 5 (hoặc thư mục notebooks/)."
sys.path.insert(0, str(root / "scripts"))
from inspect_submissions import task_registry, expected_for, inspect_task
tasks = task_registry(root)
exports = root / "submissions"
print("Repo:", root)
print("Thư mục export:", exports)


In [ ]:
semantic_names = [name for name, info in tasks.items() if info["type"] == "semantic"]
for name in semantic_names:
    result = inspect_task(name, exports / f"{name}.zip", root)
    status = "LỖI" if result["errors"] else ("CHƯA XUẤT" if not Path(result["file"]).exists() else "OK")
    print("\n", name, "·", status)
    print("  ảnh mask:", ", ".join(result["details"].get("mask_images", [])) or "—")
    for note in result["errors"]: print("  SỬA:", note)
    for note in result["warnings"]: print("  KIỂM:", note)


## Xem mask đã export

Chọn đúng task và ảnh. Màu chỉ giúp nhìn vùng; mép road/sidewalk, cột mảnh và lỗ phủ vẫn phải đối chiếu ảnh gốc bằng mắt. Nếu chưa có ZIP, ô sẽ chỉ báo bước cần làm.

In [ ]:
import io, zipfile
from IPython.display import display, Image
task_name = "easy_semantic"
zip_path = exports / f"{task_name}.zip"
if zip_path.is_file():
    with zipfile.ZipFile(zip_path) as archive:
        masks = sorted(n for n in archive.namelist() if "SegmentationClass/" in n and n.endswith(".png"))
        if masks:
            print("Mask:", masks[0])
            display(Image(data=archive.read(masks[0]), width=760))
        else: print("Không có mask SegmentationClass; kiểm lại format export.")
else: print("Chưa có ZIP:", zip_path.name)


## Câu hỏi tự QC

1. Road và sidewalk được phân theo chức năng hay màu ảnh? 2. Có vùng nhìn thấy mà chưa gán class không? 3. Nét mảnh ở `cp3_thin` đã được xem ở mức zoom lớn chưa? Ghi lỗi và hành động sửa vào `REPORT.md`.